In [1]:
import ibis

con = ibis.postgres.connect(
    user="postgres",
    password="password",
    host="postgres",
    port=5432,
    database="my_db",
)

tbl_name = "air_traffic"


In [2]:
import sys
import os

# Add project root to path
# Get the directory of this file, then go up one level to project root
current_dir = os.getcwd()
project_root = os.path.dirname(current_dir)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from sql_ai_agent.SqlAgent import SqlAgent


In [3]:
base_url = "https://api.openai.com/v1"
api_key = os.getenv("OPENAI_API_KEY")
model = "gpt-4o"
fallback_model = "gpt-5"
temperature = 0
max_token = 10000

In [4]:
agent = SqlAgent(
    api_key=api_key,
    base_url=base_url,
    model=model,
    con=con,
    fallback=True,
    fallback_model=fallback_model,
    tbl_name=tbl_name,
    memory=True,  
    memory_size=3,
    enable_logging= True
)


2026-01-20 05:19:50 - sql_ai_agent.db_handler - INFO - Query executed successfully
2026-01-20 05:19:50 - sql_ai_agent.db_handler - INFO - Query executed successfully
2026-01-20 05:19:51 - sql_ai_agent.db_handler - INFO - Query executed successfully
2026-01-20 05:19:51 - sql_ai_agent.db_handler - INFO - Query executed successfully
2026-01-20 05:19:51 - sql_ai_agent.db_handler - INFO - Query executed successfully
2026-01-20 05:19:51 - sql_ai_agent.db_handler - INFO - Query executed successfully
2026-01-20 05:19:51 - sql_ai_agent.db_handler - INFO - Query executed successfully
2026-01-20 05:19:51 - sql_ai_agent.db_handler - INFO - Query executed successfully
2026-01-20 05:19:51 - sql_ai_agent.db_handler - INFO - Query executed successfully
2026-01-20 05:19:51 - sql_ai_agent.db_handler - INFO - Query executed successfully
2026-01-20 05:19:51 - sql_ai_agent - INFO - SqlAgent initialized


In [8]:
question = "How many rows are in the dataset?"

agent.ask_question(question=question, verbose=False)


QueryOutput(success=True, validation=True, query='SELECT COUNT(*) FROM "air_traffic" LIMIT 10000', data=   count
0  38546, error=None)

In [9]:
print(agent.chat_history)

Human: How many rows are in the dataset?
AI: SELECT COUNT(*) FROM "air_traffic";


In [10]:
question = "How many passengers landed during 2024?"
agent.ask_question(question=question, verbose=False)

QueryOutput(success=True, validation=True, query='SELECT SUM("Passenger Count") FROM "air_traffic" WHERE "Activity Type Code" = \'Deplaned\' AND EXTRACT(YEAR FROM "Date") = 2024 LIMIT 10000', data=        sum
0  26079194, error=None)

In [11]:
print(agent.chat_history)


Human: How many rows are in the dataset?
AI: SELECT COUNT(*) FROM "air_traffic";
Human: How many passengers landed during 2024?
AI: SELECT SUM("Passenger Count") FROM "air_traffic" WHERE "Activity Type Code" = 'Deplaned' AND EXTRACT(YEAR FROM "Date") = 2024;


In [12]:
question = "And departure?"
agent.ask_question(question=question, verbose=False)


QueryOutput(success=True, validation=True, query='SELECT SUM("Passenger Count") FROM "air_traffic" WHERE "Activity Type Code" = \'Enplaned\' AND EXTRACT(YEAR FROM "Date") = 2024 LIMIT 10000', data=        sum
0  26054586, error=None)

In [ ]:
print(agent.chat_history)

Human: How many rows are in the dataset?
AI: SELECT COUNT(*) FROM "air_traffic";
Human: How many passengers landed during 2024?
AI: SELECT SUM("Passenger Count") FROM "air_traffic" WHERE "Activity Type Code" = 'Deplaned' AND EXTRACT(YEAR FROM "Date") = 2024;
Human: And departure?
AI: SELECT SUM("Passenger Count") FROM "air_traffic" WHERE "Activity Type Code" = 'Enplaned' AND EXTRACT(YEAR FROM "Date") = 2024;


In [14]:
question = "How many passengers passed via terminal 1?"
agent.ask_question(question=question, verbose=False)


QueryOutput(success=True, validation=True, query='SELECT SUM("Passenger Count") FROM "air_traffic" WHERE "Terminal" = \'1\' LIMIT 10000', data=    sum
0  None, error=None)

In [15]:
print(agent.chat_history)


Human: How many passengers landed during 2024?
AI: SELECT SUM("Passenger Count") FROM "air_traffic" WHERE "Activity Type Code" = 'Deplaned' AND EXTRACT(YEAR FROM "Date") = 2024;
Human: And departure?
AI: SELECT SUM("Passenger Count") FROM "air_traffic" WHERE "Activity Type Code" = 'Enplaned' AND EXTRACT(YEAR FROM "Date") = 2024;
Human: How many passengers passed via terminal 1?
AI: SELECT SUM("Passenger Count") FROM "air_traffic" WHERE "Terminal" = '1';


In [16]:
question = "That is not the right answer, the values of the Terminal field are 'Terminal 1', 'Terminal 2', etc."
agent.ask_question(question=question, verbose=False)


QueryOutput(success=True, validation=True, query='SELECT SUM("Passenger Count") FROM "air_traffic" WHERE "Terminal" = \'Terminal 1\' LIMIT 10000', data=         sum
0  251905899, error=None)